In [1]:
import pandas as pd
import numpy as np
import os

# Load in R-exported CSV with correct CBSA joins
df = pd.read_csv("/Users/brandonsmith/DATA-510-CAPSTONE--2/DATA-510-CAPSTONE--2/data/final_data/price_changes_with_collapse_flags.csv")

# replace dots in column names with underscores for easier acess
df.columns = df.columns.str.replace('.', '_', regex = False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

(83640, 26)
cbsa            float64
metro_name_x        str
year            float64
qtr             float64
dtype: object


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_8952/1113667877.py:6: DtypeWarning: Columns (0: metro_name.x, 1: RegionName.y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/brandonsmith/DATA-510-CAPSTONE--2/DATA-510-CAPSTONE--2/data/final_data/price_changes_with_collapse_flags.csv")


In [22]:
# sanity check: confirming Austin (12420), Boise (14260), and Tampa (45294) all have data
training_cbsa_map = {'Austin': 12420.0, 'Boise': 14260.0, 'Tampa': 45294.0}

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("Collapse onset check (training cities):")
print(first_collapse.loc[first_collapse.index.isin(training_cbsa_map.values())])
# this helps catches silent join failures before they cause problem in feature engineering

Collapse onset check (training cities):
                             metro_name_x    year  qtr  price_to_income_ratio
cbsa                                                                         
12420.0  Austin-Round Rock-San Marcos, TX  2021.0  2.0               5.194781
14260.0                    Boise City, ID  2019.0  3.0               5.046829
45294.0                  Tampa, FL (MSAD)  2021.0  4.0               5.213635


In [3]:
# testing to see how collapse onset date shifts if the threshold is 4.0, 4.5, 5.0 or 5.5
# Goal: confirm that 5.0 isn't an arbitry choise -- checck if a different threshold changes results signifcantly
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code in training_cbsa_map.items():
        sub = df[df['cbsa'] == code].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Results: 5.0 gives the tightest, most realistic cluster of onset dates for all three training cities


--- Threshold: 4.0 ---
  Austin: 2014.0Q3.0
  Boise: 2016.0Q2.0
  Tampa: 2017.0Q4.0

--- Threshold: 4.5 ---
  Austin: 2020.0Q4.0
  Boise: 2017.0Q4.0
  Tampa: 2021.0Q2.0

--- Threshold: 5.0 ---
  Austin: 2021.0Q2.0
  Boise: 2019.0Q3.0
  Tampa: 2021.0Q4.0

--- Threshold: 5.5 ---
  Austin: 2021.0Q3.0
  Boise: 2020.0Q4.0
  Tampa: 2022.0Q2.0


In [24]:
df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
g = df.groupby('cbsa')
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
) # final check: confirm onset dates still match austin 2021q2, boise 2019q3, Tampa, 2021q4
print(first_collapse.loc[first_collapse.index.isin(training_cbsa_map.values())])

                             metro_name_x    year  qtr  price_to_income_ratio
cbsa                                                                         
12420.0  Austin-Round Rock-San Marcos, TX  2021.0  2.0               5.194781
14260.0                    Boise City, ID  2019.0  3.0               5.046829
45294.0                  Tampa, FL (MSAD)  2021.0  4.0               5.213635


# Feature Enginnering
Every feature below is lagged by 4 quarters (1 year) before any rolling calcuation. This ensures no feature accidentally "sees" the same-quarter data used to build collapse_onset. 

# Rules
* never let a feature use current or future infro that overlaps with label construction

In [26]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Feature 1: price-to-income 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(
    lambda x: x - x.shift(20)
)

# Feature 2: affordability momentum -- 3yr rolling slope of LAGGED zhvi_yoy
df['zhvi_yoy_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(4))
df['zhvi_qoq_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(1))
df['zhvi_yoy_lag'] = df.groupby('cbsa')['zhvi_yoy_fixed'].shift(LAG)
df['affordability_momentum'] = df.groupby('cbsa')['zhvi_yoy_lag'].transform(
    lambda x: x.rolling(12, min_periods=12).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if y.notna().all() else np.nan
    )
)
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq_fixed'].shift(LAG)

# Feature 3: hpi-based momentum (yoy change in index_sa)
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)

# Secondary Signal (not in core feature set)
df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100

# Feature 4 & 5: population velocity and accleration , lagged
# Note: population coverage has a known gap for Tampa 
# These features are used for Austin/Boise and for the broader
# holdoutset, but not for unified Tampa-includsive model.
df["pop_yoy_fixed"] = g["population"].transform(lambda x: x.pct_change(4))
df["pop_velocity_fixed"] = df.groupby("cbsa")["pop_yoy_fixed"].diff()
df["pop_velocity_lag"] = df.groupby("cbsa")["pop_velocity_fixed"].shift(LAG)
df["pop_acceleration_fixed"] = df.groupby("cbsa")["pop_velocity_fixed"].diff()
df["pop_acceleration_lag"] = df.groupby("cbsa")["pop_acceleration_fixed"].shift(LAG)



In [27]:
# Coverage check

feature_cols = ['affordability_momentum', 'zhvi_qoq_lag', 'price_to_income_5yr_chg', 'hpi_yoy_lag']
pop_features = ['pop_velocity_lag', 'pop_acceleration_lag']

print("\nTraining city coverage -- 4 core features:")
for city, code in training_cbsa_map.items():
    sub = df[df["cbsa"] == code]
    coverage = {col: int(sub[col].notna().sum()) for col in feature_cols}
    print(f"  {city}: {coverage}")

print("\nTraining city coverage -- population features:")
for city, code in training_cbsa_map.items():
    sub = df[df["cbsa"] == code]
    print(f"  {city}: pop_velocity_lag={sub['pop_velocity_lag'].notna().sum()}, "
          f"pop_acceleration_lag={sub['pop_acceleration_lag'].notna().sum()}")

holdout_check = df[~df["cbsa"].isin(training_cbsa_map.values())]
print("\nHoldout coverage -- 4 core features:")
for col in train_features:
    nonnull = holdout_check[col].notna()
    print(f"  {col}: {nonnull.sum()} rows, {holdout_check[nonnull]['cbsa'].nunique()} cities")

print("\nHoldout coverage -- population features:")
for col in pop_features:
    nonnull = holdout_check[col].notna()
    print(f"  {col}: {nonnull.sum()} rows, {holdout_check[nonnull]['cbsa'].nunique()} cities")

print("\nHoldout price_to_income_ratio non-null:", holdout_check["price_to_income_ratio"].notna().sum())
print("Holdout collapse_onset True count:", holdout_check["collapse_onset"].sum())



Training city coverage -- 4 core features:
  Austin: {'affordability_momentum': 85, 'zhvi_qoq_lag': 99, 'price_to_income_5yr_chg': 40, 'hpi_yoy_lag': 132}
  Boise: {'affordability_momentum': 81, 'zhvi_qoq_lag': 95, 'price_to_income_5yr_chg': 40, 'hpi_yoy_lag': 132}
  Tampa: {'affordability_momentum': 85, 'zhvi_qoq_lag': 99, 'price_to_income_5yr_chg': 40, 'hpi_yoy_lag': 132}

Training city coverage -- population features:
  Austin: pop_velocity_lag=55, pop_acceleration_lag=54
  Boise: pop_velocity_lag=55, pop_acceleration_lag=54
  Tampa: pop_velocity_lag=0, pop_acceleration_lag=0

Holdout coverage -- 4 core features:
  affordability_momentum: 0 rows, 0 cities
  zhvi_qoq_lag: 0 rows, 0 cities
  price_to_income_5yr_chg: 0 rows, 0 cities
  hpi_yoy_lag: 12804 rows, 97 cities

Holdout coverage -- population features:
  pop_velocity_lag: 19805 rows, 371 cities
  pop_acceleration_lag: 19434 rows, 371 cities

Holdout price_to_income_ratio non-null: 0
Holdout collapse_onset True count: 0


# Findings
* price_to_income_ratio, and therefor collapse_onset as TRUE
* label, exists ONLY for Austin, Boise, and Tampa - no other metro
* has grounded-truth collpase lables in this data set. hii_yoy_lag and the population features DO have broad multi-city coverage, so they can be used to SCORE risk in other metros, but any resulting scores cannot be validated against a true label
* this remains consistent with the early-warning framing of this research question

In [28]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)


# Model 1 data: explantory (3 training cities, 4 features)
# filtering the rows for the 3 training cities with no missing feature/target values
model1_df = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=train_features + ["collapse_onset"]
).copy()

# running total of collapse_onset per city, used to split pre vs post-collapse periods
model1_df["cum_onset"] = model1_df.groupby("cbsa")["collapse_onset"].cumsum()

# training set: only rows before the first collapse_onset flag (cum_onset still 0)
model1_train = model1_df[model1_df["cum_onset"] == 0]

# validation set: rows at or after the collapse_onset event (cum_onset > 0)
model1_val = model1_df[model1_df["cum_onset"] > 0]

# sanity check
print("Model 1 -- training rows (pre-collapse, 3 cities):", len(model1_train))
print("Model 1 -- validation rows (collapse onset+, 3 cities):", len(model1_val))

# save both splits to csvs for reuse in modeling steps
model1_train.to_csv("output/model1_train.csv", index=False)
model1_val.to_csv("output/model1_val.csv", index=False)

Model 1 -- training rows (pre-collapse, 3 cities): 58
Model 1 -- validation rows (collapse onset+, 3 cities): 62


In [29]:
# Model 2 data: generalization/scoring (Austin+Boise, 3 features)
# reduce feature set used for the scoring model (drops one feature vs. model 1)
scoring_features = ["hpi_yoy_lag", "pop_velocity_lag", "pop_acceleration_lag"]
# only austin and boise this time (tampa excluded for this model)
model2_train_cities = {"Austin": training_cbsa_map["Austin"], "Boise": training_cbsa_map["Boise"]}

# filter to austin+ boise rows with no missing feature/target values
model2_df = df[df["cbsa"].isin(model2_train_cities.values())].dropna(
    subset=scoring_features + ["collapse_onset"]
).copy()

# running total of collapse_onset per city, used to split pre- vs post-collapse periods
model2_df["cum_onset"] = model2_df.groupby("cbsa")["collapse_onset"].cumsum()

# training set: rows before each city's first collapse_onset flag
model2_train = model2_df[model2_df["cum_onset"] == 0]

# validation set: rows at or after collapse_onset
model2_val = model2_df[model2_df["cum_onset"] > 0]

# quick sanity check
print("\nModel 2 -- training rows (Austin+Boise pre-collapse):", len(model2_train))
print("Model 2 -- validation rows (Austin+Boise collapse onset+):", len(model2_val))

# save both splits to CSv for resuse in modeling steps
model2_train.to_csv("output/model2_train.csv", index=False)
model2_val.to_csv("output/model2_val.csv", index=False)


Model 2 -- training rows (Austin+Boise pre-collapse): 63
Model 2 -- validation rows (Austin+Boise collapse onset+): 45


In [30]:
# holdout score set: ( all other meros, 3 features, no labels)
# exclude the 3 training cities (tampa, austin, boise) - everything else is a candidate for scoring
holdout_scoring = df[~df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=scoring_features
).copy()

# quicky sanity check on holdout set size and city coverage
print("\nHoldout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

# save holdout set for later scoring with the trained model
holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)

# confirm all output files were written
print("\nSaved: model1_train.csv, model1_val.csv, model2_train.csv, model2_val.csv, holdout_scoring.csv")


Holdout scoring rows: 3470 | cities: 65

Saved: model1_train.csv, model1_val.csv, model2_train.csv, model2_val.csv, holdout_scoring.csv


# Phase 5: modeling (xgboost) + shap explainability

In [47]:
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import(
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)

In [45]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

train_features = ["affordability_momentum", "zhvi_qoq_lag", "price_to_income_5yr_chg", "hpi_yoy_lag"]
score_features = ["hpi_yoy_lag", "pop_velocity_lag", "pop_acceleration_lag"]
target = "collapse_onset"

In [33]:
# loading in data
model1_train = pd.read_csv("output/model1_train.csv")
model1_val = pd.read_csv("output/model1_val.csv")
model2_train = pd.read_csv("output/model2_train.csv")
model2_val = pd.read_csv("output/model2_val.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

In [42]:
# model 1 - explaintoy model (austin/boise/tampa, 4 features) - small sample (58 train / 62 val),
# so the model shallow to avoid overfitting. this model is for shap explainability, not for generating production risk scores

from sklearn.model_selection import train_test_split

target = "is_unaffordable"
model1_pool = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(subset=train_features + [target]).copy()

X_all = model1_pool[train_features]
y_all = model1_pool[target].astype(int)

Xa_train, Xa_val, ya_train, ya_val = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

print("Fresh split — train:", ya_train.value_counts().to_dict())
print("Fresh split — val:", ya_val.value_counts().to_dict())
print("Overlap check:", ya_val.index.isin(ya_train.index).sum())
print("Matches old model1_val?", Xa_val.equals(model1_val[train_features]) if 'model1_val' in dir() else "n/a")

model1_fresh = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh)
})
print(classification_report(ya_val, pred_fresh))

Fresh split — train: {0: 52, 1: 32}
Fresh split — val: {0: 23, 1: 13}
Overlap check: 0
Matches old model1_val? False

Fresh Model 1 metrics:
{'accuracy': 0.8611111111111112, 'precision': 0.8333333333333334, 'recall': 0.7692307692307693, 'f1': 0.8, 'roc_auc': 0.8795986622073579}
              precision    recall  f1-score   support

           0       0.88      0.91      0.89        23
           1       0.83      0.77      0.80        13

    accuracy                           0.86        36
   macro avg       0.85      0.84      0.85        36
weighted avg       0.86      0.86      0.86        36



In [43]:
import os
os.makedirs("output/tables", exist_ok=True)

model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=is_unaffordable)",
    "accuracy": 0.8611111111111112,
    "precision": 0.8333333333333334,
    "recall": 0.7692307692307693,
    "f1": 0.8,
    "roc_auc": 0.8795986622073579
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

Saved output/tables/model1_metrics_final.csv


In [51]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

print("shap_values shape:", np.array(shap_values).shape)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": train_features,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

shap_values shape: (84, 4)
Saved output/figures/shap_summary_model1.png
                   feature  mean_abs_shap
3              hpi_yoy_lag       1.531585
2  price_to_income_5yr_chg       0.603384
0   affordability_momentum       0.518510
1             zhvi_qoq_lag       0.445252
